# c4_90—Publish snapshot to Cloud Storage

**ROI maintainers only. This is not for students.** Running it as an attendee throws a permissions
error on the upload, which is expected—you do not have write access to our bucket.

It exists to kill a single point of failure. `c4_01_load_explore.ipynb` pulls live from six public
publishers. If any one of them is down, slow, or rate-limiting at 9am on event day, it fails for
the whole room at once. `scripts/load.sh` rebuilds the identical tables from a Cloud Storage
snapshot instead, and this notebook is what produces that snapshot.

## The rule that makes it trustworthy

**This notebook does not reimplement the student notebook.** It fetches it by raw URL, executes it
once per state with `STATE` and `DATASET` overridden, and exports what lands in BigQuery. That
guarantees the fallback cannot drift from what students actually get—which it silently would, the
first time somebody fixed a bug in one file and not the other.

Two design choices worth knowing about, both learned the expensive way on earlier challenges:

- **We export from BigQuery, not from dataframes in memory.** The student notebook enriches
  `shelters` with a county via a `CREATE OR REPLACE TABLE` *after* loading it, so the in-memory
  frame is missing two columns the finished table has. Reading back from BigQuery means the
  snapshot is by construction the notebook's end state.
- **We gate on the student notebook's own validation**, read back out of the executed namespace,
  and we refuse to publish a state whose checks did not run at all. A suite reporting "no failures"
  because it never executed is the most dangerous green there is.

In [ ]:
BUCKET       = "class-demo"
PREFIX       = "a4i-2026/challenge-4-evacuation"
NOTEBOOK_URL = ("https://raw.githubusercontent.com/haggman/"
                "A4I2026-challenge-4-evacuation/main/notebooks/c4_01_load_explore.ipynb")

# Never publish out of the dataset a human has been poking at.
SCRATCH_DATASET = "a4i_c4_publish_scratch"

STATES = ["FL", "TX", "GA", "NC", "SC", "LA", "NY", "CA"]

TABLES = ["shelters", "vulnerability_tracts", "hazard_tracts",
          "care_facilities", "power_dependent_counties"]
OPTIONAL_TABLES = {"care_facilities", "power_dependent_counties"}

# The student notebook runs 20 checks today. Demand most of them rather than an exact
# count, so adding one does not break publishing, but deleting the section does.
MIN_CHECKS = 16

import os, subprocess
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or subprocess.check_output(
    ["gcloud", "config", "get-value", "project"], text=True).strip()
print(f"project : {PROJECT_ID}")
print(f"target  : gs://{BUCKET}/{PREFIX}/<STATE>/<table>/data.parquet")
print(f"states  : {', '.join(STATES)}")

## Fetch the student notebook and extract its code

Two cells are skipped by marker: the Section 2 hook cells, which each spend a Grounding with Google
Maps call to make a teaching point. Eight states times four grounded prompts is thirty-two calls
that teach nobody anything and cost real money.

Then we locate the config cell **and prove it is unambiguous** before touching it. The regex is the
part most likely to rot: a rename in the student notebook that stops the pattern matching would make
`re.subn` substitute nothing, report nothing, and publish eight identical copies of Florida. That
has happened on this project before, so the substitution count is asserted and both overrides are
test-fired on a probe before the long run starts.

In [ ]:
import re
import requests
import nbformat

SKIP_MARKER = "c4_90_publish_snapshot.ipynb skips this cell"

nb = nbformat.reads(requests.get(NOTEBOOK_URL, timeout=60).text, as_version=4)

sources, skipped = [], {"magic": 0, "marked": 0}
for cell in nb.cells:
    if cell.cell_type != "code":
        continue
    src = cell.source
    if src.lstrip().startswith("%%"):          # exec() cannot run cell magics
        skipped["magic"] += 1
        continue
    if SKIP_MARKER in src:
        skipped["marked"] += 1
        continue
    sources.append(src)

print(f"{len(sources)} executable cells; skipped {skipped['magic']} magics, "
      f"{skipped['marked']} marked")
if skipped["marked"] != 2:
    raise RuntimeError(
        f"Expected to skip exactly 2 marked cells (the two Section 2 hook cells); skipped "
        f"{skipped['marked']}. Either the markers were removed or a new one was added. "
        f"Check before publishing—an unskipped hook cell costs a grounded prompt per state, "
        f"and a wrongly-skipped one means the snapshot came from a different notebook.")

# --- Find the config cell, once, and prove it is unambiguous ----------------
# STATE_FIPS is assigned in the same cell and must NOT match. The pattern requires
# whitespace-or-equals immediately after STATE, and STATE_FIPS has an underscore
# there, so it is excluded by construction. Verified below rather than assumed.
STATE_LINE   = re.compile(r'(?m)^STATE\s*=\s*.+$')
DATASET_LINE = re.compile(r'(?m)^DATASET\s*=\s*[^\n]+')

config_idx = [i for i, s in enumerate(sources)
              if STATE_LINE.search(s) and DATASET_LINE.search(s) and "STATE_FIPS" in s]
if len(config_idx) != 1:
    raise RuntimeError(
        f"Expected exactly one config cell assigning both STATE and DATASET; found "
        f"{len(config_idx)}: {config_idx}. The student notebook's config cell has changed "
        f"shape and the overrides below would not work. Fix this before publishing anything.")
CONFIG_IDX = config_idx[0]

assert not STATE_LINE.search('STATE_FIPS = {"AL": "01"}'), \
    "STATE pattern is too loose—it would rewrite STATE_FIPS"


def configure(src, state):
    """Rewrite STATE and DATASET in the config cell. Raises rather than no-ops."""
    src, n = STATE_LINE.subn(lambda _: f'STATE = "{state}"', src, count=1)
    if n != 1:
        raise RuntimeError(f"STATE substitution matched {n} lines, expected 1")
    src, n = DATASET_LINE.subn(lambda _: f'DATASET = "{SCRATCH_DATASET}"', src, count=1)
    if n != 1:
        raise RuntimeError(f"DATASET substitution matched {n} lines, expected 1")
    return src


# Prove both substitutions work before spending ten minutes finding out they do not.
probe = configure(sources[CONFIG_IDX], "TX")
assert 'STATE = "TX"' in probe, "state override did not take"
assert f'DATASET = "{SCRATCH_DATASET}"' in probe, "dataset override did not take"
assert 'STATE_FIPS = {' in probe, "the FIPS lookup was clobbered by the STATE substitution"
print(f"Config cell is #{CONFIG_IDX}; both overrides test-fired successfully.")

## Test the gate before trusting it

The gate is the only thing standing between a broken run and a snapshot that 150 people load on
event day, so it gets tested rather than assumed.

Earlier challenges used a deliberately thin instance as a negative control. That works, but it
tests the gate only indirectly and depends on knowing an instance that fails. Here we hand the gate
five synthetic namespaces that *should* be rejected and confirm each one is. It is faster, it is
deterministic, and it tests the actual failure branches rather than hoping one gets exercised.

Then two that *should* be accepted: a clean run, and a run carrying WARNs. The second matters.
The student notebook distinguishes **our load being broken** (FAIL) from **the publisher's data
being untidy** (WARN), and the first version of this gate treated anything that was not a PASS as
disqualifying. That withheld Texas, Georgia and California over four odd rows in nine thousand.
A gate that is too strict fails silently in the direction of shipping nothing, which is easy to
mistake for working correctly.

In [ ]:
def gate(ns, state):
    """Raise if this run must not be published. Every branch here is tested below."""
    ran_as = ns.get("STATE")
    if ran_as != state:
        raise RuntimeError(
            f"state override failed: asked for {state!r}, notebook ran as {ran_as!r}. "
            f"Nothing uploaded. Fix configure() before rerunning.")

    if ns.get("DATASET") != SCRATCH_DATASET:
        raise RuntimeError(
            f"{state} validated against DATASET={ns.get('DATASET')!r}, not "
            f"{SCRATCH_DATASET!r}. Those checks graded the wrong tables.")

    checks = ns.get("CHECKS")
    if not checks:
        raise RuntimeError(
            "the validation section produced no checks—it did not run. Refusing to publish "
            "something nobody checked.")
    if len(checks) < MIN_CHECKS:
        raise RuntimeError(
            f"only {len(checks)} checks ran, expected at least {MIN_CHECKS}. The validation "
            f"section is incomplete.")
    # PASS and WARN both publish; only FAIL blocks. A WARN says the source publisher's
    # data is untidy, which is true of every state and is teaching material rather than
    # a defect in the load. Treating the two the same is what cost us TX, GA and CA on
    # the first publish run: four odd rows in nine thousand, three states withheld.
    failed = [ck for ck in checks if ck.get("result") == "FAIL"]
    if failed:
        raise RuntimeError("validation failed: " +
                           "; ".join(f"{ck['check']} ({ck['detail']})" for ck in failed))
    warned = [ck for ck in checks if ck.get("result") == "WARN"]
    if warned:
        print(f"  {len(warned)} WARN (source data, not our load): " +
              "; ".join(ck["check"] for ck in warned))


def _ns(state="FL", dataset=None, checks=None):
    return {"STATE": state, "DATASET": dataset or SCRATCH_DATASET, "CHECKS": checks}


_ok = [{"check": f"c{i}", "result": "PASS", "detail": ""} for i in range(MIN_CHECKS)]
_bad = _ok[:-1] + [{"check": "boom", "result": "FAIL", "detail": "on purpose"}]

NEGATIVE_CONTROLS = {
    "wrong state":        (_ns(state="TX"), "FL"),
    "wrong dataset":      (_ns(dataset="evacuation_readiness"), "FL"),
    "no checks at all":   (_ns(checks=None), "FL"),
    "too few checks":     (_ns(checks=_ok[:3]), "FL"),
    "a failing check":    (_ns(checks=_bad), "FL"),
}

for label, (ns, state) in NEGATIVE_CONTROLS.items():
    try:
        gate(ns, state)
    except RuntimeError as exc:
        print(f"  rejected as it should be—{label:<18} {str(exc)[:60]}")
    else:
        raise RuntimeError(
            f"THE GATE IS BROKEN: '{label}' was accepted. Do not publish anything from this "
            f"run until it is fixed—a gate that passes everything is worse than no gate.")

# Both positive controls. The second one is the change made 2026-08-10 and is the whole
# reason TX, GA and CA publish at all, so it gets tested rather than assumed.
_warn = _ok[:-1] + [{"check": "untidy source", "result": "WARN", "detail": "2 odd rows"}]
gate(_ns(checks=_ok), "FL")
gate(_ns(checks=_warn), "FL")
print(f"\nGate accepts a clean run and a run carrying WARNs, and rejects all "
      f"{len(NEGATIVE_CONTROLS)} negative controls.")

## Build, validate, then export—one state at a time

Each state runs the whole student notebook end to end into a scratch dataset, passes the gate, and
is then exported straight out of BigQuery.

A failure on one state does not stop the others. The summary at the bottom is what you read.

In [ ]:
import io, time, traceback
import pandas as pd
from google.cloud import bigquery, storage

# The student notebook calls display() in several cells. exec() runs in a namespace we
# build by hand, and that namespace does not inherit the notebook's injected builtins—
# so display resolves to nothing and every table cell raises NameError. Pass it in.
try:
    from IPython.display import display as _display
except Exception:                                   # not in a notebook at all
    _display = print

NOTEBOOK_BUILTINS = {"display": _display}

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET)
bqc = bigquery.Client(project=PROJECT_ID)

results = {}

for state in STATES:
    print(f"\n{'=' * 68}\n{state}\n{'=' * 68}")
    t0 = time.time()
    ns = {"__name__": "__main__", **NOTEBOOK_BUILTINS}
    try:
        for i, src in enumerate(sources):
            if i == CONFIG_IDX:
                src = configure(src, state)
            exec(compile(src, f"<cell {i}>", "exec"), ns)

        gate(ns, state)

        # Does the DATA say it is this state, or only the config? Row counts cannot tell
        # eight states apart; a modal state code and a FIPS prefix can.
        fips = ns["STATE_FIPS"][state]
        row = list(bqc.query(f"""
            SELECT (SELECT COUNT(*) FROM `{PROJECT_ID}.{SCRATCH_DATASET}.shelters`) n,
                   (SELECT COUNTIF(state = '{state}')
                      FROM `{PROJECT_ID}.{SCRATCH_DATASET}.shelters`) right_state,
                   (SELECT COUNTIF(SUBSTR(geo_id, 1, 2) = '{fips}')
                      FROM `{PROJECT_ID}.{SCRATCH_DATASET}.vulnerability_tracts`) right_fips,
                   (SELECT COUNT(*) FROM `{PROJECT_ID}.{SCRATCH_DATASET}.vulnerability_tracts`) t
        """).result())[0]
        if row.right_state != row.n:
            raise RuntimeError(
                f"{row.n - row.right_state} of {row.n} shelters do not carry state='{state}'")
        if row.right_fips != row.t:
            raise RuntimeError(
                f"{row.t - row.right_fips} of {row.t} tracts do not start with FIPS {fips}")

        # --- Export, straight out of BigQuery ---------------------------------
        written = []
        for table in TABLES:
            try:
                df = bqc.query(
                    f"SELECT * FROM `{PROJECT_ID}.{SCRATCH_DATASET}.{table}`").to_dataframe()
            except Exception:
                if table in OPTIONAL_TABLES:
                    print(f"  {table:<28} absent—skipped (optional)")
                    continue
                raise
            if not len(df):
                if table in OPTIONAL_TABLES:
                    print(f"  {table:<28} empty—skipped (optional)")
                    continue
                raise RuntimeError(f"required table {table} was empty")
            buf = io.BytesIO()
            df.to_parquet(buf, index=False)
            buf.seek(0)
            bucket.blob(f"{PREFIX}/{state}/{table}/data.parquet").upload_from_file(
                buf, content_type="application/octet-stream")
            written.append(f"{table}({len(df):,})")
        print(f"  uploaded: {', '.join(written)}")
        results[state] = ("OK", f"{row.n:,} shelters, {row.t:,} tracts, "
                                f"{time.time() - t0:.0f}s")

    except Exception as exc:                                       # noqa: BLE001
        results[state] = ("FAILED", str(exc)[:300])
        print(f"  FAILED: {exc}")
        traceback.print_exc(limit=3)

print(f"\n{'=' * 68}\nSUMMARY\n{'=' * 68}")
for state, (status, detail) in results.items():
    print(f"{status:<8} {state:<4} {detail}")

## Verify the snapshot is loadable—and that no two states are copies

Row counts cannot tell eight states apart. A publish loop with a broken override produces eight
perfectly valid, perfectly identical snapshots, and every row count looks plausible.

So we read each state's parquet back out of Cloud Storage and fingerprint it: the modal state code,
the tract FIPS prefix, and a hash of the shelter ID set. If two fingerprints match, something
published the same state twice.

In [ ]:
import hashlib

problems, fingerprints, rows = [], {}, []

for state in STATES:
    if results.get(state, ("", ""))[0] != "OK":
        continue
    try:
        uri = f"gs://{BUCKET}/{PREFIX}/{state}/shelters/data.parquet"
        df = pd.read_parquet(uri)

        modal = df["state"].mode().iat[0] if "state" in df.columns else "?"
        ids = sorted(str(x) for x in df["shelter_id"])
        fp = hashlib.sha256("|".join(ids).encode()).hexdigest()[:16]

        if modal != state:
            problems.append(f"{state}: shelters say state={modal!r}")
        if fp in fingerprints:
            problems.append(f"{state}: identical shelter set to {fingerprints[fp]}—"
                            f"the override did not take for one of them")
        fingerprints[fp] = state

        wc_null = int(df["wheelchair_accessible"].isna().sum())
        rows.append({"state": state, "shelters": len(df), "fingerprint": fp,
                     "capacity": int(df["evacuation_capacity"].fillna(0).sum()),
                     "wheelchair_unrecorded": wc_null,
                     "unrecorded_pct": round(100 * wc_null / max(len(df), 1)),
                     "medical": int(df["is_medical"].sum())})
    except Exception as exc:                                       # noqa: BLE001
        problems.append(f"{state}: could not read back—{exc}")

display(pd.DataFrame(rows))

if problems:
    print("\nPROBLEMS")
    for p in problems:
        print(f"  {p}")
    raise RuntimeError(f"{len(problems)} snapshot problem(s)—do not announce this snapshot.")
print("\nEvery published state is distinct and is the state it claims to be.")

## Finally: the summary

Nothing below this line can fail in a way that matters. That is deliberate.

A previous challenge's publisher uploaded and verified ten instances correctly and then threw a
permissions error in its last cell, because it called an API it never needed. A clean publish looked
like a failed run. **A final reporting cell must not be able to fail after the work has succeeded**,
and it must not attempt a check the notebook is structurally unable to perform—a publisher running
as a maintainer cannot prove a student's read access. Only `bash scripts/load.sh` in a fresh Skills
project can do that, and that is a separate step on the Stage 3 checklist.

In [ ]:
print("=" * 70)
print("C4 PUBLISH—SUMMARY")
print("=" * 70)
ok = [s for s, (st, _) in results.items() if st == "OK"]
bad = [s for s, (st, _) in results.items() if st != "OK"]
print(f"published : {len(ok)}/{len(STATES)}  {', '.join(ok)}")
if bad:
    print(f"FAILED    : {', '.join(bad)}")
print(f"location  : gs://{BUCKET}/{PREFIX}/<STATE>/<table>/data.parquet")
print()
print("The bucket needs allAuthenticatedUsers:objectViewer—NOT allUsers.")
print("load.sh never makes an HTTP request: it uses `gcloud storage` and `bq load`")
print("against gs:// URIs, and a BigQuery load job reads the source object as the")
print("job submitter. Every attendee is signed in inside the lab, so authenticated")
print("access covers them and the bucket does not need opening to the world.")
print()
print("NEXT, and it is the only thing that proves any of this worked:")
print("  run `bash scripts/load.sh FL` in a FRESH Skills project, then run it again.")
print("  The second run is the one that matters—it takes the other branch.")
print("=" * 70)

try:
    for state in ok:
        blobs = list(gcs.list_blobs(BUCKET, prefix=f"{PREFIX}/{state}/"))
        size = sum(b.size or 0 for b in blobs) / 1e6
        print(f"  {state:<4} {len(blobs):>2} objects, {size:6.1f} MB")
except Exception as exc:                                           # noqa: BLE001
    print(f"  (object listing unavailable: {type(exc).__name__}—the upload above still "
          f"succeeded, this line is cosmetic)")